# Reproducing Schiavo et al. (2020) from dandiset 000249 and its figshare recordings

Paper. Schiavo JK, Valtcheva S, Bair-Marshall CJ, Song SC, Martin KA, Froemke RC (2020).
Innate and plastic mechanisms for maternal behaviour in auditory cortex. *Nature* 587, 426-431.
[doi:10.1038/s41586-020-2807-6](https://doi.org/10.1038/s41586-020-2807-6). The claims file was
written from the NIH author manuscript in PubMed Central (PMC7677212), with its Methods and the
Extended Data legends; this attempt read the same text.

Dandiset. [DANDI:000249](https://dandiarchive.org/dandiset/000249/0.230423.1416), version
0.230423.1416 (doi:10.48324/dandi.000249/0.230423.1416). The paper cites no DANDI version (its data
statement points to a figshare DOI). This is the only published version, so it is pinned. The
draft lists the same 777 assets and size; it was modified later (2024-05-18), and the analysis
uses nothing that a draft could change.

What the dandiset holds. 777 NWB files of 54 mice, each holding one raw, unaligned two-photon
movie (`TwoPhotonSeries`), the subject id, a session description that names the stimulus set
(for example `base_r1_pc1_71b_j6a_61f_002`), and an experiment description that names the group.
There are no ROIs, fluorescence traces, stimulus times, stimulus order or trial tables. The paper's
behavioural assays, whole-cell recordings and vocalization recordings are not in the dandiset.

Data from elsewhere. The dandiset's metadata links the figshare collection
doi:10.6084/m9.figshare.c.5043830.v6 (the paper's own figshare DOI, 10.35092/yhjc.c.5037737, does
not resolve). The claims file marks the whole-cell, slice and operant claims `data: elsewhere`, at
the figshare articles it names, which are read here at their versions: in vivo whole-cell
recordings of experienced (12611657 v3) and naive females (12613496 v2), slice recordings with
repeated stimulation (12611522 v1), oxytocin slice recordings (12613811 v1), operant touch times
(13056584 v1) and the operant task code (13054076 v1). The imaging items of the collection hold the
same unaligned movies as TIFF files, with the same descriptions, and add nothing that the
dandiset lacks.

Claims. Tested here: `subject-group-link`, `fig1g-ipsc-depress-faster`,
`fig1i-operant-prototypes`, `ed5b-press-count-unchanged`, `ed6ef-epsc-tuning`,
`fig2h-epsc-enhanced-ev`, `fig2i-ipsc-weaker-nv`, `fig2j-ie-ratio-tracks-isi`,
`fig2k-ipsc-exceeds-epsc-575` and `ed9b-oxytocin-enhances-spiking`. Blocked:

| Claims | Reason |
|---|---|
| fig1a-isi-distribution, fig1c-warm-vs-cold, fig1c-salient-isis-retrieved, fig1c-slow-and-single-not-retrieved, ed2a-dams-evs-similar, ed2b-latency-slow-longer, ed2d-ymaze-approach, fig4c-ch575-retrieve-slow, fig4d-actx-inhibition-blocks, ed8hi-single-syllables-not-learned, fig4e-ot-inhibition-deficits | data missing: the vocalization recordings and the retrieval and y-maze behaviour are neither in the dandiset nor in the figshare collection |
| ed4d-response-tracks-isi-probability | data missing: it needs the ISI distribution of fig1a (and the imaging responses, below) |
| ed3e, fig1f, ed6c, ed3k, ed3m, ed4c, ed4e, ed4g, fig2d, ed6d, fig2e, fig2f, ed7bc, fig3e, fig3h, ed8cd, fig4b, fig4g, fig4h, ed3bc (all imaging claims) | metadata insufficient: the movies carry no stimulus times and no order of the calls within a recording, and no ROIs |
| fig3-retrieval-onset-range | metadata insufficient: the start of cohousing is not recorded |
| ed6h-dams-evs-epsc-similar | metadata insufficient: the whole-cell spreadsheets do not say which experienced cells come from dams |

Two tested claims end blocked as `methods_underspecified` because the readings the claims file
names give different outcomes (`fig1i-operant-prototypes`, `ed5b-press-count-unchanged`), and one
for the same reason over the 5 or 10 largest trials (`fig2i-ipsc-weaker-nv`); see their sections.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM

import schiavo as S

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
print("dandiset", S.DANDISET_ID, "version", S.DANDISET_VERSION)
print("figshare articles (id, version):", S.FIGSHARE)

## Data check

### The dandiset

Every NWB file of the pinned version is opened (streamed); the survey reads each file's groups,
the movie's shape, type and rate, the descriptions, and the first and last frame (to compare
recordings). Nothing else of the movies is read here.

In [ ]:
meta = S.dandi_version_metadata()
summary = meta["assetsSummary"]
print("Archive summary: files", summary["numberOfFiles"], "| subjects", summary["numberOfSubjects"],
      "| bytes", summary["numberOfBytes"], "| variables", summary.get("variableMeasured"))
print("Related resources:", [(r.get("relation"), r.get("identifier")) for r in meta.get("relatedResource", [])])
assets = S.asset_table()
nwb = S.nwb_summaries(assets)
files = assets.merge(nwb, on="path")
print(f"\nFiles read: {len(files)}; subjects: {files.subject_id.nunique()}")
for col in ["acquisition", "processing", "stimulus", "intervals", "analysis"]:
    print(f"{col:>12}:", files[col].astype(str).value_counts().to_dict())
print("\nMovie shapes, types and rates:")
print(files.groupby(["height", "width", "dtype", "rate", "imaging_rate"]).size().to_string())
print("\nIndicator and location fields:", files.indicator.unique().tolist(), files.location.unique().tolist())
print("Frames per movie: min", files.n_frames.min(), "median", int(files.n_frames.median()), "max", files.n_frames.max())
print("Session description in the file equals the archive's:",
      bool((files.session_description_file == files.session_description).all()))
print("Subject id in the file equals the archive's:", bool((files.subject_id == files.subject).all()))

Each file holds one movie and nothing else: no `processing` module (no ROIs or traces), no
stimulus series and no intervals (trials). The movies are 256 x 256 pixels at 3.90625 Hz, which
matches the Methods' 256 x 256 pixels at "4 Hz (0.26 s/frames)". Indicator and location are
"unknown" in every file, although the experiment descriptions name the indicator class and the
hemisphere.

In [ ]:
groups = (files.groupby("subject_id").experiment_description.first()
          .apply(S.group_from_description).apply(pd.Series))
groups["n_files"] = files.groupby("subject_id").size()
groups["description_names"] = files.groupby("subject_id").experiment_description.first().str.extract(
    r"videos for\s*\n?\s*([A-Za-z0-9-]+)", expand=False)
groups["tone_sequences"] = files.groupby("subject_id").session_description.apply(
    lambda s: bool(s.str.contains("kmtf", case=False).any()))
groups["tones_only"] = files.groupby("subject_id").session_description.apply(
    lambda s: bool(s.str.contains("tone|kmtf", case=False).all()))
groups["first_retrieval"] = files.groupby("subject_id").session_description.apply(
    lambda s: bool(s.str.lower().str.replace(" ", "").str.contains("firstretrieval").any()))
groups["ch575"] = files.groupby("subject_id").experiment_description.first().str.contains("CH\\+575")
groups["not_cohoused_control"] = files.groupby("subject_id").experiment_description.first().str.contains("not cohoused")
print(groups.to_string())
mismatch = groups[groups.description_names.str.upper() != groups.index.str.upper()]
print("\nSubjects whose experiment description names another id:")
print(mismatch[["description_names"]].to_string())

### Recordings deposited more than once

Exact copies would share a checksum. Copies under a new encoding would share their frame count and
the sums of their first and last frames.

In [ ]:
print("Files sharing a SHA-256 digest:", int(files.sha256.duplicated(keep=False).sum()))
key = files.groupby(["n_frames", "frame0_sum", "frame_last_sum"]).path.apply(list)
dups = key[key.map(len) > 1]
for paths in dups:
    sub = files[files.path.isin(paths)][["path", "session_start", "session_description", "n_frames", "size"]]
    print(sub.to_string(index=False), "\n")
print("Recordings deposited twice:", len(dups))

Three recordings (the `plus24hrs_postexp1` sessions of 2018-03-08) are deposited under both
sub-NV58 and sub-NV63, with the same start times and the same frames. Nothing in the files says to
which mouse they belong. They are counted once below.

### Is the timing of the calls recoverable from the movies?

The calls were "played in a pseudorandom order (repetition rate: 0.1 Hz)" (Methods). With no
stimulus times, a test of any imaging claim would need the onset and the identity of each call
from the movie itself. The spectrum of the movie (8 x 8 pixel blocks, each as dF/F, high-passed
with a 20 s moving average) is shown for a prototype file, a morph file and a pure tone file.

In [ ]:
from scipy.ndimage import uniform_filter1d

def block_spectrum(s3):
    import h5py, remfile
    with h5py.File(remfile.File(s3), "r") as f:
        x = f["acquisition/TwoPhotonSeries/data"][:].astype(np.float32)
        fs = float(f["acquisition/TwoPhotonSeries/starting_time"].attrs["rate"])
    b = x.reshape(len(x), 32, 8, 32, 8).mean(axis=(2, 4)).reshape(len(x), -1)
    b = (b - b.mean(0)) / b.mean(0)
    b = b - uniform_filter1d(b, int(20 * fs), axis=0)
    P = (np.abs(np.fft.rfft(b, axis=0)) ** 2).mean(axis=1)
    return np.fft.rfftfreq(len(b), 1 / fs), P

examples = ["sub-NV43/sub-NV43_ses-20171014T183731_ophys.nwb",   # pc1: three prototypes
            "sub-EV27/sub-EV27_ses-20170425T192357_ophys.nwb",  # morph set of 61f,
            "sub-NV33/sub-NV33_ses-20170711T155914_ophys.nwb"]   # pure tones
examples = [p for p in examples if p in set(files.path)] or files.path.iloc[:3].tolist()
fig, ax = plt.subplots(figsize=(8, 3))
for p in examples:
    row = files[files.path == p].iloc[0]
    fr, P = block_spectrum(row.s3)
    sel = (fr > 0.05) & (fr < 1.0)
    rel = P[sel] / np.median(P[sel])
    near = (fr[sel] > 0.09) & (fr[sel] < 0.11)
    top = np.argsort(rel)[::-1][:3]
    print(f"{row.session_description}: power at 0.09-0.11 Hz {rel[near].max():.1f} x median; "
          f"largest peaks at {np.round(fr[sel][top], 3).tolist()} Hz ({np.round(rel[top], 1).tolist()} x median)")
    ax.semilogy(fr[sel], rel, label=row.session_description[:40])
ax.axvline(0.1, color="grey", lw=0.5)
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("power / median"); ax.legend(fontsize=7)
plt.show()

In the call files the power near the 0.1 Hz call rate (3.8 and 6.2 times the median) does not
stand above the slower fluctuations (8.9 and 10.4 times the median at 0.06 to 0.07 Hz), so the
onsets cannot be read from the movie's periodicity, and nothing in a file gives the order of the
three to five calls it names.
Every imaging claim needs responses assigned to calls, so all are blocked as
`metadata_insufficient`. The pure tone file (tones at 0.2 Hz, Methods) shows peaks near 0.195 and
0.39 Hz, which may be the tones or a scanning artefact at a twentieth of the frame rate.

### The figshare recordings

The in vivo whole-cell articles list each cell, file, stimulus set, holding potential and notes in
a spreadsheet. Each ABF file has the membrane current and a microphone channel (`MIC`) that
records the played call, from which the call's onset, offset and ISI are measured per sweep.

In [ ]:
iv_list, iv_sweeps, iv_info = S.load_invivo()
print("Files listed:", len(iv_list), "| on figshare:", int(iv_list.on_figshare.sum()))
print("Listed but not on figshare:", iv_list[~iv_list.on_figshare][["group", "cell", "file", "set", "holding"]].to_dict("records"))
print("\nCells per group:", iv_list.groupby("group").cell.nunique().to_dict(),
      "(paper: 12 experienced, 14 naive EPSC cells; 6 and 6 IPSC cells)")
print("Recording settings:", iv_info.groupby(["rate", "sweep_s", "protocol"]).size().to_dict())
print("Channels:", iv_info.adc.astype(str).unique().tolist())
sets = iv_list[~iv_list.set.str.contains("5xcalls") & iv_list.on_figshare].file
sw = iv_sweeps[iv_sweeps.file.isin(sets)]
print("\nMorph-set files:", sw.file.nunique(), "| sweeps:", len(sw), "| sweeps with no call found:", int(sw.isi.isna().sum()))
print("Files whose ISI comes from call-duration ranks:", int(sw.groupby("file").duration_rank_used.first().sum()),
      "| from the measured gap only:", sw.groupby("file").duration_rank_used.first().eq(False).sum())
print("\nPer ISI bin, measured silent gap (ms) and call duration (s):")
print(sw.dropna(subset=["isi"]).groupby("isi").agg(sweeps=("gap_ms", "size"), gap_min=("gap_ms", "min"),
      gap_median=("gap_ms", "median"), gap_max=("gap_ms", "max"), dur_min=("duration", "min"),
      dur_max=("duration", "max")).round(3).to_string())
agree = (sw.isi == sw.isi_gap).mean()
print(f"Sweeps where the duration rank and the nearest gap bin agree: {agree:.1%}")
order = sw.dropna(subset=["isi"]).groupby("file").isi.apply(lambda v: "-".join(str(int(i)) for i in v.iloc[:4]))
print("Order of the first four sweeps, by file:", order.value_counts().to_dict())
print("Call onset within the 5 s sweep (s): min %.2f, max %.2f" % (sw.onset.min(), sw.onset.max()))

In [ ]:
hc = iv_sweeps.groupby("file").holding_current.median().rename("median_holding_current_pA")
chk = iv_list.merge(hc, left_on="file", right_index=True)
chk["listed"] = np.where(chk.holding < -30, "-70 mV", "0 to +60 mV")
print("Median holding current (pA) by listed holding potential:")
print(chk.groupby("listed").median_holding_current_pA.describe().round(1).to_string())
odd = chk[((chk.holding < -30) & (chk.median_holding_current_pA > 100)) |
          ((chk.holding > -30) & (chk.median_holding_current_pA < -100))]
print("\nFiles whose holding current contradicts the listed potential:")
print(odd[["group", "cell", "file", "set", "holding", "median_holding_current_pA", "notes"]].to_string(index=False))
per_sweep = iv_sweeps.merge(iv_list[["file", "holding"]], on="file")
bad = per_sweep[((per_sweep.holding < -30) & (per_sweep.holding_current > 100)) |
                ((per_sweep.holding > -30) & (per_sweep.holding_current < -100))]
print("Sweeps dropped for this, by file:", bad.groupby("file").size().to_dict())

In [ ]:
# Do the currents follow the calls? Per cell and current, the signed evoked current (call window
# minus the 1 s before onset) over trials, against zero; the sham measure (the 0.75 s before onset
# minus the 0.75 s before that) is the same test with the window moved before the call.
_, raw_ev = S.psc_table(iv_list, iv_sweeps, top_n=5)
_, raw_sh = S.psc_table(iv_list, iv_sweeps, top_n=5, measure="sham")
rows = []
for name, raw in (("call window", raw_ev), ("sham, before the call", raw_sh)):
    for (grp, cell, kind), d in raw.groupby(["group", "cell", "kind"]):
        tt = stats.ttest_1samp(d.evoked, 0)
        expected = tt.statistic < 0 if kind == "EPSC" else tt.statistic > 0
        rows.append({"window": name, "group": grp, "cell": cell, "kind": kind, "trials": len(d),
                     "mean pA": d.evoked.mean(), "p": tt.pvalue, "responds": bool(expected and tt.pvalue < 0.05)})
resp = pd.DataFrame(rows)
print("Cells whose mean current over trials moves in the expected direction at p < 0.05:")
print(resp.groupby(["window", "kind", "group"]).responds.agg(["sum", "size"]).to_string())
print("\nCall window, per cell:")
print(resp[resp.window == "call window"].drop(columns="window").round(3).to_string(index=False))
sd = iv_sweeps.merge(iv_list[["file", "group", "holding"]], on="file").dropna(subset=["baseline_sd"])
sd["kind"] = np.where(sd.holding < -30, "EPSC", "IPSC")
print("\nBaseline current s.d. within a sweep (pA), median by group and current:")
print(sd.groupby(["kind", "group"]).baseline_sd.median().round(2).to_string())

The microphone channel gives every sweep of the 82 complete morph-set files a call, and in each of
them the four call durations fall in four separate groups that repeat in a fixed order (prototype,
125, 275, 575 ms); the median measured silent gaps of the four groups are 142, 192, 301 and 604 ms.
The stimulus therefore follows the legends' ISIs (125, 175, 275, 575 ms), not the Methods' "125,
225, 575". The order is fixed, not pseudorandom. File 19424013 is listed at -70 mV but carries
about +560 pA throughout, as recorded at a depolarized potential; 19411001 is listed at -70 mV
and its note says "some sweeps are at positive potentials". Sweeps whose current contradicts the
listed potential are dropped (the counts are printed above); keeping them is a variant below.

In [ ]:
sl_files, _ = S.figshare_files("slice")
sl_list = S.slice_list(pd.read_excel(__import__("io").BytesIO(S.fetch(sl_files["Recording List.xlsx"]["download_url"])), header=None))
sl_list["on_figshare"] = sl_list.file.map(lambda f: f + ".abf" in sl_files)
print("Slice recordings listed:", len(sl_list), "| on figshare:", int(sl_list.on_figshare.sum()))
print("Missing:", sl_list[~sl_list.on_figshare].to_dict("records"))
from concurrent.futures import ThreadPoolExecutor
def _slice(f):
    return S.slice_amplitudes(S.fetch(sl_files[f + ".abf"]["download_url"]), f + ".abf")
with ThreadPoolExecutor(8) as ex:
    sl_amps = pd.concat(list(ex.map(_slice, sl_list[sl_list.on_figshare].file)))
sl_amps = sl_amps.merge(sl_list, on="file")
print("Cells per rate:", sl_list[sl_list.on_figshare].groupby("rate").cell.nunique().to_dict(), "(paper: 75 ms 10, 175 ms 14, 575 ms 13)")
print("Pulse periods (ms) found in the files, by rate bin:")
print(sl_amps.groupby("rate").period_ms.unique().apply(lambda v: sorted(np.round(v, 1))).to_string())
print("Holding levels of the two trains (mV):", sorted(sl_amps.level.unique()))
print("Pulses per train:", sorted(sl_amps.pulse.unique()))
per_trial = sl_amps[sl_amps.sweep >= 0]
print("Trials (sweeps) per cell and rate: median", int(per_trial.groupby(["cell", "rate", "file"]).sweep.nunique().groupby(["cell", "rate"]).sum().median()))
print("First-pulse amplitudes (pA) by current:")
print(per_trial[per_trial.pulse == 1].groupby("kind").amp.describe().round(1).to_string())

Each slice sweep holds one train of five pulses at -70 mV (EPSCs) and one at a depolarized level
(IPSCs; 0 mV in most files, -10 to -30 mV in some, within the Methods' "-40 to 0 mV"). The pulse
periods (50 and 87.5 ms; 150 and 175 ms; 550 and 575 ms) fall in the Methods' 75, 175 and 575 ms
bins of plus or minus 25 ms, except 50 ms, at the edge of the 75 ms bin. One 575 ms cell (Cell 20,
file 19o11005) is listed but not on figshare, so 12 of 13 cells are tested at 575 ms.

In [ ]:
ox_files, _ = S.figshare_files("oxytocin")
ox_sheet = [n for n in ox_files if n.endswith(".xlsx")][0]
ox_list = S.oxytocin_list(pd.read_excel(__import__("io").BytesIO(S.fetch(ox_files[ox_sheet]["download_url"])), header=None))
ox_list["on_figshare"] = ox_list.file.map(lambda f: f + ".abf" in ox_files)
print("Oxytocin files listed:", len(ox_list), "| on figshare:", int(ox_list.on_figshare.sum()))
print("Cells:", ox_list.groupby("group").cell.nunique().to_dict(), "(paper: 9 oxytocin, 7 ACSF)")
print("Roles from the spreadsheet labels:", ox_list.groupby(["group", "role"]).size().to_dict())
def _ox(f):
    df, start = S.spikes_per_pulse(S.fetch(ox_files[f + ".abf"]["download_url"]), f + ".abf")
    df["file_start"] = start
    return df
with ThreadPoolExecutor(8) as ex:
    ox_sp = pd.concat(list(ex.map(_ox, ox_list[ox_list.on_figshare & (ox_list.role != "iv")].file)))
ox_sp = ox_sp.merge(ox_list[["file", "group", "cell", "role", "label"]], on="file")
ox_sp["abs_t"] = pd.to_datetime(ox_sp.file_start) + pd.to_timedelta(ox_sp.t, unit="s")
print("Pulse period (ms):", sorted(ox_sp.period_ms.round(2).unique()), "| pulses per sweep:",
      sum(c.startswith("stim") for c in ox_sp.columns), "| sweep interval (s):", sorted(ox_sp.groupby("file").t.diff().dropna().round(1).unique()))
print("Current steps (pA):", sorted(ox_sp.step_pA.unique()))

In [ ]:
op = S.operant_files()
with ThreadPoolExecutor(16) as ex:
    op["touches"] = list(ex.map(S.read_touches, op.url))
op["n_samples"] = op.touches.map(len)
op["first_s"] = op.touches.map(lambda v: v.min() if len(v) else np.nan)
op["last_s"] = op.touches.map(lambda v: v.max() if len(v) else np.nan)
op["interval_ms"] = op.touches.map(S.sampling_interval) * 1000
print("Operant files:", len(op), "| mice per group:", op.groupby("group").mouse.nunique().to_dict(), "(paper: 7 per group)")
print("Sessions per mouse:", op.groupby("mouse").session.nunique().value_counts().to_dict(),
      "| missing sessions:", {m: sorted(set(range(1, 9)) - set(g.session)) for m, g in op.groupby("mouse") if set(range(1, 9)) - set(g.session)})
print("Part b files joined to their session:", op[op.part_b].name.tolist())
print("Touch times (s): first %.2f to %.2f, last %.1f to %.1f (sessions run 1800 s after a start offset)"
      % (op.first_s.min(), op.first_s.max(), op.last_s.min(), op.last_s.max()))
print("Sampling interval between touch samples (ms, from times below 100 s): median %.3f, range %.3f-%.3f"
      % (op.interval_ms.median(), op.interval_ms.min(), op.interval_ms.max()))
allt = np.concatenate(op.touches.values)
for lo, hi in [(0, 100), (100, 1000), (1000, 3000)]:
    v = allt[(allt >= lo) & (allt < hi)]
    d = np.diff(np.unique(v))
    print(f"times {lo}-{hi} s: smallest step between distinct values {d.min():.4f} s; "
          f"share of samples repeating the previous value {np.mean(np.diff(v) == 0):.2f}")

The touch times are printed with five significant digits (MATLAB `dlmwrite`'s default), so they
step by 1 ms below 100 s, 10 ms to 1000 s and 100 ms after that, and most samples repeat the value
before them. A press can only be resolved to that step: a gap must exceed the local step to end a
press (the helper's `presses`). The task code polls the sensor about every 0.7 ms.

## subject-group-link

> The dandiset's subjects can be assigned to the paper's imaging groups (experienced or naive
> virgin, dam, right cortex, cohousing, CH+575, opto-OT; excitatory, inhibitory or pan-neuronal
> indicator), with stimulus identity (prototype or morph ISI) for each presentation.

In [ ]:
g = groups.copy()
g["paper_group"] = "other"
g.loc[(g.group == "EV") & (g.cells == "excitatory") & (g.hemisphere == "left"), "paper_group"] = "EV excitatory, left"
g.loc[(g.group == "EV") & (g.cells == "inhibitory"), "paper_group"] = "EV interneurons"
g.loc[(g.group == "EV") & (g.hemisphere == "right"), "paper_group"] = "EV right cortex"
g.loc[g.group == "dam", "paper_group"] = "dam"
g.loc[g.cells.str.startswith("pan"), "paper_group"] = "CH+575 opto-OT"
g.loc[g.ch575 & ~g.cells.str.startswith("pan"), "paper_group"] = "CH+575"
g.loc[(g.group == "NV") & (g.cells == "inhibitory") & g.cohoused, "paper_group"] = "NV interneurons, chronic cohousing"
g.loc[(g.group == "NV") & (g.cells == "excitatory") & g.cohoused & ~g.ch575 & ~g.not_cohoused_control, "paper_group"] = "NV excitatory, chronic cohousing"
g.loc[g.not_cohoused_control, "paper_group"] = "NV excitatory, not cohoused (control)"
tone_only = files.groupby("subject_id").session_description.apply(lambda s: bool(s.str.contains("tone|kmtf", case=False).all()))
g.loc[tone_only.reindex(g.index).fillna(False).astype(bool), "paper_group"] = "tones only (no pup calls)"
counts = g.groupby("paper_group").apply(lambda d: pd.Series({"n": len(d), "subjects": ", ".join(sorted(d.index))}))
print(counts.to_string())
paper = {"EV excitatory, left": 9, "EV interneurons": 5, "EV right cortex": 4, "dam": 1, "CH+575": 6,
         "CH+575 opto-OT": 4, "NV interneurons, chronic cohousing": 6}
print("\nAgainst the paper's counts:")
for k, v in paper.items():
    print(f"  {k}: dandiset {int((g.paper_group == k).sum())}, paper {v}")
chronic = g[g.first_retrieval]
print(f"  chronic cohousing (subjects with a First Retrieval session): dandiset {len(chronic)} "
      f"({', '.join(chronic.index)}), paper 13")
nv_exc = g[(g.group == "NV") & (g.cells == "excitatory")]
print(f"  naive excitatory mice of any kind: {len(nv_exc)}; paper's Fig. 1f NV excitatory N=12 (which ones is not said)")
print("\nSubjects without a group:", g[g.paper_group == "other"].index.tolist())
print("Subjects whose experience group could not be read:", g[g.group == "unknown"].index.tolist())
print("\nStimulus identity per presentation: stimulus series in any file:",
      int((files.stimulus.map(len) > 0).sum()), "| interval tables:", int((files.intervals.map(len) > 0).sum()))

**Result: partially reproduced.** The experiment descriptions place 49 of the 54 subjects in
the paper's groups, and the counts match the paper for EV excitatory in left cortex (9), EV
interneurons (5), EV right cortex (4), the dam (1), CH+575 (6), CH+575 opto-OT (4) and naive
interneurons (6). 14 subjects have a first retrieval session, against the paper's 13 chronically
imaged virgins, and 20 naive excitatory mice of several kinds stand against the 12 of Fig. 1f,
which the paper does not name. For the five mice imaged only with tones the experience group
cannot be read: EV56 and EV62 are named NV56 and NV62 in their own descriptions. No file holds a
stimulus series or a trial table, so no presentation can be assigned its call or ISI, which the
criterion makes a partial result. Three recordings are deposited under both NV58 and NV63.

## fig1g-ipsc-depress-faster

> In pup-naive auditory cortical slices, IPSCs depress faster than EPSCs at the prototypical
> stimulation rate (175 ms), while EPSCs and IPSCs depress similarly at 75 or 575 ms.

Method from the Methods ("The peak amplitude of evoked EPSCs and IPSCs were measured and
normalized to the event evoked by the first stimulation on each trial (Sn/S1)") and the claims
file: per cell the mean Sn/S1 over trials; IPSC against EPSC by one-way ANOVA (with two groups the
same test as an unpaired t-test) under reading (a), per pulse with Bonferroni over pulses 2 to 5,
and (b), the mean over pulses 2 to 5. The peak is the largest inward (EPSC) or outward (IPSC)
current 2 to 30 ms after each pulse against the 5 ms before it, a standard choice stated in the
helper. A trial whose first IPSC or EPSC is not above baseline gives no ratio and is dropped; the
mean trace of each file is a variant.

In [ ]:
def stp_tests(amps, label):
    a = amps.pivot_table(index=["rate", "cell", "kind", "file", "sweep"], columns="pulse", values="amp")
    dropped = int((a[1] <= 0).sum())
    a = a[a[1] > 0]
    c = a.div(a[1], axis=0).groupby(["rate", "cell", "kind"]).mean()
    rows = []
    for rate in (75, 175, 575):
        e = c.xs((rate, "EPSC"), level=("rate", "kind"))
        i = c.xs((rate, "IPSC"), level=("rate", "kind"))
        p = np.array([stats.ttest_ind(i[k], e[k]).pvalue for k in (2, 3, 4, 5)])
        mi, me = i[[2, 3, 4, 5]].mean(axis=1), e[[2, 3, 4, 5]].mean(axis=1)
        rows.append({"variant": label, "rate": rate, "n_E": len(e), "n_I": len(i),
                     "E_S2..S5": e[[2, 3, 4, 5]].mean().round(2).tolist(), "I_S2..S5": i[[2, 3, 4, 5]].mean().round(2).tolist(),
                     "(a) p per pulse, Bonferroni": np.round(np.minimum(p * 4, 1), 3).tolist(),
                     "(a) I below E at p<0.05": bool(((p * 4) < 0.05).any() and (i[[2, 3, 4, 5]].mean() < e[[2, 3, 4, 5]].mean())[(p * 4) < 0.05].all()),
                     "(b) mean S2-5 E": round(me.mean(), 2), "(b) mean S2-5 I": round(mi.mean(), 2),
                     "(b) p": round(stats.ttest_ind(mi, me).pvalue, 4),
                     "(b) paired p": round(stats.ttest_rel(mi.loc[mi.index.intersection(me.index)], me.loc[mi.index.intersection(me.index)]).pvalue, 4)})
    return pd.DataFrame(rows), dropped

t1, d1 = stp_tests(sl_amps[sl_amps.sweep >= 0], "per trial")
t2, d2 = stp_tests(sl_amps[sl_amps.sweep < 0], "mean trace per file")
print(f"Trials dropped for a first response not above baseline: {d1} (per trial); files dropped: {d2} (mean trace)")
print(pd.concat([t1, t2]).to_string(index=False))

In [ ]:
a = sl_amps[sl_amps.sweep >= 0].pivot_table(index=["rate", "cell", "kind", "file", "sweep"], columns="pulse", values="amp")
a = a[a[1] > 0]
c = a.div(a[1], axis=0).groupby(["rate", "cell", "kind"]).mean()
fig, axs = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, rate in zip(axs, (75, 175, 575)):
    for kind, col in (("EPSC", "k"), ("IPSC", "r")):
        x = c.xs((rate, kind), level=("rate", "kind"))
        ax.errorbar(range(1, 6), x.mean(), x.sem(), color=col, label=kind)
    ax.set_title(f"{rate} ms"); ax.set_xlabel("pulse")
axs[0].set_ylabel("Sn/S1"); axs[0].legend()
axs[0].set_ylim(0, 2)
outside = int(((c < 0) | (c > 2)).sum().sum())
print("Cell means of Sn/S1 range from %.2f to %.2f; %d values fall outside the axes (0 to 2)" % (c.min().min(), c.max().max(), outside))
plt.show()

**Result: not reproduced.** At 175 ms IPSCs do not depress more than EPSCs. Per trial, the
mean Sn/S1 over pulses 2 to 5 is 0.90 for IPSCs against 0.68 for EPSCs (p = 0.33), and no pulse
differs after Bonferroni correction (all adjusted p = 1.0). On the mean trace of each file the
IPSC ratio is lower (0.57 against 0.65, p = 0.39), with pulses 2 and 3 at adjusted p 0.15 and 0.06.
At 75 ms neither reading differs (p 0.52 and 0.72 for the mean over pulses). At 575 ms the
unpaired tests do not differ (p 0.08 and 0.82), but on mean traces the paired test finds IPSCs
lower than EPSCs (0.55 against 0.66, p = 0.022), where the claim has them similar. 35 trials and 7
file means with no first response above baseline were dropped, and 17 cell means of Sn/S1 fall
outside 0 to 2, so the per-trial IPSC ratios are noisy. The outcome rests on the peak measure (2
to 30 ms after each pulse against the 5 ms before it), which the paper does not specify.

## fig1i-operant-prototypes

> Of naive virgins able to silence playback with a lever press, only those hearing prototypes
> significantly increased their press duration by session 8.

The claims file's readings: a press ends at a gap above (a) twice the median inter-sample interval
or (b) 100 ms; the comparison is (a) session 1 against session 8, or (b) the mean of sessions 1-2
against 7-8; paired two-tailed t-test per group of 7 mice. Files marked `b` are joined to their
session. The median inter-sample interval is taken per session from the part of the session
printed at 1 ms (data check); one session with no such part uses the median over sessions.

In [ ]:
glob_interval = np.nanmedian(op.touches.map(S.sampling_interval))
rows = []
for (mouse, grp, ses), d in op.groupby(["mouse", "group", "session"]):
    t = np.concatenate(d.touches.values)
    iv = np.nanmedian([S.sampling_interval(v) for v in d.touches])
    iv = glob_interval if np.isnan(iv) else iv
    for rule in ("median2", "100ms"):
        p = S.presses(t, rule, iv)
        rows.append({"mouse": mouse, "group": grp, "session": ses, "rule": rule, "presses": len(p),
                     "mean_duration_s": p[:, 2].mean() if len(p) else np.nan, "max_duration_s": p[:, 2].max() if len(p) else np.nan})
press = pd.DataFrame(rows)
print("Longest single press (s) under each rule:", press.groupby("rule").max_duration_s.max().round(1).to_dict())
print("\nMean press duration (s) per mouse and session, gap rule (b) 100 ms:")
print(press[press.rule == "100ms"].pivot_table(index=["group", "mouse"], columns="session", values="mean_duration_s").round(2).to_string())

def operant_tests(measure):
    out = []
    for rule in ("median2", "100ms"):
        for comp in ("1 vs 8", "1-2 vs 7-8"):
            for grp in ("proto", "fast", "slow"):
                x = press[(press.rule == rule) & (press.group == grp)].pivot(index="mouse", columns="session", values=measure)
                a, b = (x[1], x[8]) if comp == "1 vs 8" else (x[[1, 2]].mean(axis=1), x[[7, 8]].mean(axis=1))
                ok = a.notna() & b.notna()
                tt = stats.ttest_rel(b[ok], a[ok])
                out.append({"gap rule": rule, "comparison": comp, "group": grp, "n": int(ok.sum()),
                            "early": round(a[ok].mean(), 3), "late": round(b[ok].mean(), 3),
                            "t": round(tt.statistic, 2), "p": round(tt.pvalue, 4),
                            "increase at p<0.05": bool(tt.pvalue < 0.05 and tt.statistic > 0)})
    return pd.DataFrame(out)

dur = operant_tests("mean_duration_s")
print("\nMean press duration, session comparisons (paired two-tailed t-test):")
print(dur.to_string(index=False))
verdict = dur.groupby(["gap rule", "comparison"]).apply(
    lambda d: bool(d.set_index("group").loc["proto", "increase at p<0.05"]
                   and not d.set_index("group").loc[["fast", "slow"], "p"].lt(0.05).any()))
print("\nReading reproduces the claim (prototypes increase, the others not significant):")
print(verdict.to_string())

**Result: blocked (methods underspecified).** Prototypes increase the mean press duration
under all four readings (session 1 against 8: p = 0.025 with the median-interval gap, 0.0076
with the 100 ms gap; sessions 1-2 against 7-8: p = 0.032 and 0.021). The fast and slow groups are
not significant under three readings (p 0.081 to 0.73), but with the 100 ms gap and session 1
against 8 the slow group also increases (p = 0.040) and the fast group reaches p = 0.050. The
criterion blocks the claim when the readings disagree. The reading closest to the paper's p-values
(0.03, 0.24, 0.13) is the median-interval gap with session 1 against 8 (0.025, 0.081, 0.17).

## ed5b-press-count-unchanged

> The number of lever presses per session did not increase by session 8 in any of the three
> groups.

Presses counted as above, under both gap rules and both comparisons; paired two-tailed t-test per
group of 7.

In [ ]:
cnt = operant_tests("presses")
print(cnt.to_string(index=False))
holds = cnt.groupby(["gap rule", "comparison"])["increase at p<0.05"].apply(lambda v: not v.any())
print("\nReading finds no increase in any group:")
print(holds.to_string())

**Result: blocked (methods underspecified).** No group's number of presses increases under
three readings (all p >= 0.09). With the median-interval gap and sessions 1-2 against 7-8, the fast
group increases from 68.5 to 107.4 presses (p = 0.049). The criterion blocks the claim when the
readings disagree.

## ed6ef-epsc-tuning

> In experienced females, prototype- and morph-evoked EPSCs are similar and negatively correlated
> with ISI; in naive cortex morph-evoked EPSCs are weaker than prototype-evoked ones and
> uncorrelated with ISI.

Method (Methods, "In vivo whole-cell recordings", quoted in the helper): per sweep the mean current
from call onset to 250 ms after call offset minus the mean of the 1 s before onset; per cell and
ISI the mean absolute value of the largest 5 or 10 trials (both run). EPSCs are the files held at
-70 mV, IPSCs those at 0 to +60 mV. Experienced cells pool dams and EVs (the spreadsheet does not
separate them). Repeated-measures one-way ANOVA over the four ISIs; post hoc paired t-tests with
Bonferroni correction, over the three comparisons with the prototype that the legend reports, and
over all six pairs (a reading of the family, which the legend does not give). Pearson correlation
over all cell and ISI points. The main analysis follows the spreadsheet notes and the best call
where one is named, and drops sweeps whose holding current contradicts the listed potential; the
variant uses every sweep of every set file.

In [ ]:
def psc(top_n, main=True):
    kw = {} if main else dict(apply_notes=False, best_call_only=False, check_holding=False)
    t, raw = S.psc_table(iv_list, iv_sweeps, top_n=top_n, **kw)
    return t.pivot_table(index=["group", "cell", "kind"], columns="isi", values="value"), t, raw

tables = {(n, m): psc(n, m) for n in S.TOP_N for m in (True, False)}
w, t, raw = tables[(5, True)]
print("Trials per cell and ISI (main, EPSC): min", int(t[t.kind == "EPSC"].n_trials.min()),
      "median", int(t[t.kind == "EPSC"].n_trials.median()))
print("Share of trials whose evoked current has the expected sign (EPSC inward, IPSC outward):",
      raw.assign(ok=lambda d: np.where(d.kind == "EPSC", d.evoked < 0, d.evoked > 0)).groupby("kind").ok.mean().round(2).to_dict())
rows = []
for (n, main), (w, _, _) in tables.items():
    for grp in ("experienced", "naive"):
        e = w.xs((grp, "EPSC"), level=("group", "kind")).dropna()
        long = e.reset_index().melt(id_vars="cell", var_name="isi", value_name="v")
        an = AnovaRM(long, "v", "cell", within=["isi"]).fit().anova_table
        p3 = {c: min(1, stats.ttest_rel(e[175], e[c]).pvalue * 3) for c in (125, 275, 575)}
        p6 = {(a, b): min(1, stats.ttest_rel(e[a], e[b]).pvalue * 6) for i, a in enumerate(e.columns) for b in e.columns[i + 1:]}
        r = stats.pearsonr(long.isi.astype(float), long.v)
        rows.append({"top": n, "sweeps": "main" if main else "all", "group": grp, "cells": len(e),
                     "mean 125/175/275/575": e.mean().round(1).tolist(),
                     "RM ANOVA F": round(an["F Value"].iloc[0], 2), "RM p": round(an["Pr > F"].iloc[0], 4),
                     "175 vs 125/275/575, Bonf x3": [round(v, 3) for v in p3.values()],
                     "smallest pair p, Bonf x6": round(min(p6.values()), 3),
                     "Pearson r": round(r.statistic, 3), "r p": round(r.pvalue, 4), "points": len(long)})
ed6ef = pd.DataFrame(rows)
print(ed6ef.to_string(index=False))

In [ ]:
w, _, _ = tables[(5, True)]
print("Per-cell EPSC (pA, largest 5 trials, main):")
print(w.xs("EPSC", level="kind").round(1).to_string())
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, grp in zip(axs, ("experienced", "naive")):
    e = w.xs((grp, "EPSC"), level=("group", "kind"))
    for _, r in e.iterrows():
        ax.plot(e.columns, r.values, color="0.7", lw=0.8)
    ax.errorbar(e.columns, e.mean(), e.sem(), color="k")
    ax.set_title(f"{grp} (n={len(e)})"); ax.set_xlabel("ISI (ms)")
axs[0].set_ylabel("|EPSC| (pA)")
print("Plotted range: %.1f to %.1f pA, all points shown" % (w.xs("EPSC", level="kind").min().min(), w.xs("EPSC", level="kind").max().max()))
plt.show()

**Result: not reproduced.** In naive cells the prototype-evoked EPSC is not larger than the
morph-evoked ones after Bonferroni correction: with the largest 5 trials, 20.9 pA against 17.4,
17.2 and 18.3 pA (adjusted p 0.057, 0.20 and 0.83; repeated-measures ANOVA p = 0.21), and no reading
gives p < 0.05 for all three comparisons (one reading gives 0.048 for 125 ms alone). Experienced
EPSCs are not correlated with ISI (r = -0.16, p = 0.28; r -0.12 to -0.16 under every reading).
The parts that hold: no experienced pair differs (smallest adjusted p 0.086 to 0.52 over all six
pairs) and the naive correlation is not significant (r = -0.01, p = 0.92). The data check finds
that the trial-averaged EPSC follows the call at p < 0.05 in only 3 of 26 cells, so the Methods'
measure (the largest 5 to 10 absolute magnitudes) here mostly picks the largest excursions of
the background current.

## fig2h-epsc-enhanced-ev

> Prototype-evoked EPSCs are similar between groups, but EPSCs evoked by non-prototypical ISIs are
> larger in experienced than in naive females.

EPSC magnitudes as in ed6ef-epsc-tuning; experienced against naive per ISI by unpaired one-tailed
t-test (experienced larger), as the legend gives; the two-tailed p is shown as well.

In [ ]:
rows = []
for (n, main), (w, _, _) in tables.items():
    E = w.xs("EPSC", level="kind")
    for c in (125, 175, 275, 575):
        a, b = E.xs("experienced", level="group")[c].dropna(), E.xs("naive", level="group")[c].dropna()
        rows.append({"top": n, "sweeps": "main" if main else "all", "isi": c, "EV mean": round(a.mean(), 1), "n EV": len(a),
                     "NV mean": round(b.mean(), 1), "n NV": len(b), "ratio": round(a.mean() / b.mean(), 2),
                     "one-tailed p": round(stats.ttest_ind(a, b, alternative="greater").pvalue, 4),
                     "two-tailed p": round(stats.ttest_ind(a, b).pvalue, 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nNoise control: the same measure taken before the call (sham), largest 5 trials, main sweeps:")
sham = S.psc_table(iv_list, iv_sweeps, top_n=5, measure="sham")[0].pivot_table(index=["group", "cell", "kind"], columns="isi", values="value")
for kind in ("EPSC", "IPSC"):
    Z = sham.xs(kind, level="kind")
    for c in (125, 175, 275, 575):
        a, b = Z.xs("experienced", level="group")[c].dropna(), Z.xs("naive", level="group")[c].dropna()
        print(f"  {kind} {c} ms: EV {a.mean():.1f} (n {len(a)}), NV {b.mean():.1f} (n {len(b)}), "
              f"one-tailed p {stats.ttest_ind(a, b, alternative='greater').pvalue:.4f}, two-tailed p {stats.ttest_ind(a, b).pvalue:.4f}")

**Result: partially reproduced.** Experienced EPSCs exceed naive ones at 125, 275 and 575 ms
(one-tailed p 0.0013, 0.0008 and 0.021 with the largest 5 trials; 0.0007 to 0.021 under every
reading), but also at the prototype (32.5 against 20.9 pA, p = 0.020; 0.009 to 0.020 under every
reading), where the paper finds them similar. The same measure taken in the 0.75 s before the
call gives the same group difference at every ISI (EV 30.7 to 34.7 against NV 15.3 to 19.8 pA,
one-tailed p 0.0003 to 0.012), and the within-sweep s.d. of the baseline current is 11.3 pA in
experienced against 5.2 pA in naive cells. The difference found here is therefore present before
any call and does not show an enhanced call-evoked EPSC.

## fig2i-ipsc-weaker-nv

> IPSCs are broadly tuned in both groups but weaker in naive virgins at every ISI.

IPSC magnitudes as in ed6ef-epsc-tuning (cells with a set file held at 0 to +60 mV); experienced
against naive per ISI by unpaired two-tailed t-test; Pearson correlation of IPSC with ISI over cell
and ISI points per group.

In [ ]:
rows, cors = [], []
for (n, main), (w, _, _) in tables.items():
    I = w.xs("IPSC", level="kind")
    for c in (125, 175, 275, 575):
        a, b = I.xs("experienced", level="group")[c].dropna(), I.xs("naive", level="group")[c].dropna()
        rows.append({"top": n, "sweeps": "main" if main else "all", "isi": c, "EV mean": round(a.mean(), 1), "n EV": len(a),
                     "NV mean": round(b.mean(), 1), "n NV": len(b), "two-tailed p": round(stats.ttest_ind(a, b).pvalue, 4)})
    for grp in ("experienced", "naive"):
        x = I.xs(grp, level="group").dropna().reset_index().melt(id_vars="cell", var_name="isi", value_name="v")
        r = stats.pearsonr(x.isi.astype(float), x.v)
        cors.append({"top": n, "sweeps": "main" if main else "all", "group": grp, "cells": x.cell.nunique(),
                     "r": round(r.statistic, 3), "p": round(r.pvalue, 4)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(pd.DataFrame(cors).to_string(index=False))
w, _, _ = tables[(5, True)]
print("\nIPSC cells (main):", {g: sorted(w.xs((g, "IPSC"), level=("group", "kind")).index) for g in ("experienced", "naive")})

**Result: blocked (methods underspecified).** Experienced IPSCs exceed naive ones at 125, 275
and 575 ms under every reading (p 0.0018 to 0.027). At 175 ms they do with the largest 10 trials
(p = 0.040 and 0.039) but not with the largest 5 (p = 0.061 under both sweep selections). The
Methods say "the largest 5-10 evoked trials", so neither is the paper's choice, and the outcomes
differ. Neither group's IPSC is correlated with ISI (r -0.09 to 0.13, p 0.51 to 0.75), as the
claim says. The measure taken before the call shows the same group difference (EV 34.2 to 45.6
against NV 12.1 to 13.8 pA, two-tailed p 0.0011 to 0.039; printed under fig2h). Seven experienced
cells have IPSC recordings, where the paper has six.

## fig2j-ie-ratio-tracks-isi

> In experienced females the IPSC-EPSC ratio is positively correlated with ISI, so that the
> balance shifts toward inhibition for slower ISIs; not so in naive females.

(IPSC - EPSC)/(IPSC + EPSC) per cell and ISI (Methods), from cells with both currents, correlated
with ISI over all cell and ISI points.

In [ ]:
rows = []
for (n, main), (w, _, _) in tables.items():
    for grp in ("experienced", "naive"):
        both = w.xs(grp, level="group").unstack("kind").dropna()
        E, I = both.xs("EPSC", axis=1, level="kind"), both.xs("IPSC", axis=1, level="kind")
        ratio = (I - E) / (I + E)
        x = ratio.reset_index().melt(id_vars="cell", var_name="isi", value_name="v")
        r = stats.pearsonr(x.isi.astype(float), x.v)
        rows.append({"top": n, "sweeps": "main" if main else "all", "group": grp, "cells": len(ratio),
                     "mean ratio 125/175/275/575": ratio.mean().round(2).tolist(), "r": round(r.statistic, 3), "p": round(r.pvalue, 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** In the 7 experienced cells with both currents the IPSC-EPSC ratio
is not correlated with ISI (r = 0.063, p = 0.75; r 0.063 to 0.087 under every reading), against
the paper's r = 0.47 (tolerance 0.42 to 0.52). The naive correlation is not significant (r =
0.077, p = 0.72), as in the paper.

## fig2k-ipsc-exceeds-epsc-575

> Within cells of experienced cortex, 575 ms-evoked IPSCs are stronger than EPSCs, while the two
> do not differ at 125, 175 or 275 ms.

Experienced cells with both currents; paired two-tailed t-test of IPSC against EPSC per ISI.

In [ ]:
rows = []
for (n, main), (w, _, _) in tables.items():
    both = w.xs("experienced", level="group").unstack("kind").dropna()
    E, I = both.xs("EPSC", axis=1, level="kind"), both.xs("IPSC", axis=1, level="kind")
    for c in (125, 175, 275, 575):
        rows.append({"top": n, "sweeps": "main" if main else "all", "isi": c, "cells": len(both),
                     "IPSC": round(I[c].mean(), 1), "EPSC": round(E[c].mean(), 1),
                     "cells with I > E": int((I[c] > E[c]).sum()), "paired p": round(stats.ttest_rel(I[c], E[c]).pvalue, 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("Cells:", sorted(both.index))

**Result: not reproduced.** At 575 ms IPSCs are larger than EPSCs on average (41.0 against
31.7 pA with the largest 5 trials, larger in 4 of 7 cells) but not significantly (paired p = 0.29;
0.17 to 0.48 under every reading). At 125, 175 and 275 ms they do not differ (p 0.23 to 0.96), as
in the paper.

## ed9b-oxytocin-enhances-spiking

> Oxytocin enhances spiking to repeated 575 ms pulses in auditory cortical slices from stimulus 2
> on, while stimulation without oxytocin does not change spike probability.

Spike probability per pulse (a crossing of 0 mV before the next pulse, a standard choice) at
baseline (the files labelled baseline or PRE) and over the sweeps 15 to 30 minutes after wash
onset (Methods), the onset being the first sweep of the first "wash in" file; for ACSF controls,
"following baseline acquisition" (Methods) is read as from the first sweep after the baseline
files. Friedman test over the ten columns (baseline and post for each stimulus) with Dunn's
comparisons of baseline against post for each stimulus, Bonferroni over the five (the legend
gives one p per stimulus); all 45 pairs and Wilcoxon signed-rank tests are variants.

In [ ]:
stims = [f"stim{k}" for k in range(1, 6)]
rows = []
for (grp, cell), d in ox_sp.groupby(["group", "cell"]):
    d = d.sort_values("abs_t")
    base = d[d.role == "baseline"]
    if not len(base):
        base = d[d.file == d.file.iloc[0]]
    onset = d[d.role == "wash in"].abs_t.min() if grp == "oxytocin" else d[~d.file.isin(base.file)].abs_t.min()
    mins = (d.abs_t - onset).dt.total_seconds() / 60
    post = d[(mins >= S.OXY_WINDOW_MIN[0]) & (mins <= S.OXY_WINDOW_MIN[1])]
    r = {"group": grp, "cell": cell, "baseline sweeps": len(base), "post sweeps": len(post),
         "post files": ",".join(sorted(post.file.unique()))}
    for s in stims:
        r["base " + s] = base[s].mean()
        r["post " + s] = post[s].mean() if len(post) else np.nan
    rows.append(r)
ox = pd.DataFrame(rows)
print(ox.round(2).to_string(index=False))
for grp in ("oxytocin", "control"):
    x = ox[ox.group == grp].dropna(subset=["post stim1"])
    data = np.c_[x[["base " + s for s in stims]].values, x[["post " + s for s in stims]].values]
    chi, dunn = S.dunn_friedman(data, [(k, k + 5) for k in range(5)])
    _, dunn45 = S.dunn_friedman(data, [(i, j) for i in range(10) for j in range(i + 1, 10)])
    sel45 = dunn45.set_index("pair").loc[[(k, k + 5) for k in range(5)]]
    print(f"\n{grp}: {len(x)} cells with sweeps in the window (of {ox[ox.group == grp].shape[0]}); "
          f"Friedman chi2 = {chi.statistic:.2f}, p = {chi.pvalue:.2g}")
    out = pd.DataFrame({"stim": stims, "baseline": x[["base " + s for s in stims]].mean().values.round(3),
                        "post": x[["post " + s for s in stims]].mean().values.round(3),
                        "Dunn p, Bonf x5": dunn.p_bonf.round(4).values,
                        "Dunn p, Bonf x45": (sel45.p * 45).clip(upper=1).round(4).values,
                        "Wilcoxon p": [round(stats.wilcoxon(x["post " + s], x["base " + s]).pvalue, 4)
                                       if (x["post " + s] != x["base " + s]).any() else 1.0 for s in stims]})
    print(out.to_string(index=False))

**Result: reproduced**, resting on the reading of the Dunn comparisons. In the 9 oxytocin
cells spike probability rises from 0.035 to 0.051 at baseline to 0.48 to 0.56 in the 15 to 30
minutes after wash onset (Friedman chi2 = 51.35, p = 6e-08). With Dunn's comparisons of baseline
against post for each stimulus, Bonferroni over five, stimuli 2 to 5 increase (adjusted p 0.022,
0.014, 0.0011 and 0.020) and stimulus 1 does not (p = 0.071), the pattern of the paper (0.34;
0.009, 0.008, 0.001, 0.04). The 5 ACSF controls with sweeps in the window show no change
(Friedman p = 0.56, all adjusted p >= 0.79). The outcome depends on the comparison family: over
all 45 pairs only stimulus 4 is significant (p = 0.0098), and Wilcoxon tests find stimulus 1
significant too (p = 0.0078). Two of the 7 controls have no sweeps 15 to 30 minutes after their
baseline and are left out.

## Blocked claims: what the data show

The start of cohousing is not recorded, so retrieval onset (fig3-retrieval-onset-range) cannot be
measured. The hours from the last baseline session to the first retrieval session are an upper
bound only if cohousing began after the baseline imaging, which nothing in the files says; they
are printed for the record and decide nothing.

In [ ]:
x = files.assign(t=pd.to_datetime(files.session_start.str[:19]), d=files.session_description.str.lower().str.replace(" ", ""))
rows = []
for s, d in x.groupby("subject_id"):
    fr = d[d.d.str.contains("firstretrieval")]
    if not len(fr):
        continue
    b = d[d.d.str.startswith("base")]
    rows.append({"subject": s, "last baseline": b.t.max(), "first retrieval": fr.t.min(),
                 "hours": round((fr.t.min() - b.t.max()).total_seconds() / 3600, 1) if len(b) else np.nan})
print(pd.DataFrame(rows).to_string(index=False))
print("Subjects with a first retrieval session:", len(rows), "(paper: N=13 with onset 12 to 109 h)")
print(f"\nRun time: {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| subject-group-link | with labels | partially reproduced |
| fig1g-ipsc-depress-faster | elsewhere (figshare) | not reproduced (assumed) |
| fig1i-operant-prototypes | elsewhere (figshare) | blocked: methods underspecified |
| ed5b-press-count-unchanged | elsewhere (figshare) | blocked: methods underspecified |
| ed6ef-epsc-tuning | elsewhere (figshare) | not reproduced |
| fig2h-epsc-enhanced-ev | elsewhere (figshare) | partially reproduced |
| fig2i-ipsc-weaker-nv | elsewhere (figshare) | blocked: methods underspecified |
| fig2j-ie-ratio-tracks-isi | elsewhere (figshare) | not reproduced |
| fig2k-ipsc-exceeds-epsc-575 | elsewhere (figshare) | not reproduced |
| ed6h-dams-evs-epsc-similar | elsewhere (figshare) | blocked: metadata insufficient |
| ed9b-oxytocin-enhances-spiking | elsewhere (figshare) | reproduced (assumed) |
| 20 imaging claims (ed3e, fig1f, ed6c, ed3k, ed3m, ed4c, ed4e, ed4g, fig2d, ed6d, fig2e, fig2f, ed7bc, fig3e, fig3h, ed8cd, fig4b, fig4g, fig4h, ed3bc) | dandiset | blocked: metadata insufficient |
| fig3-retrieval-onset-range | dandiset | blocked: metadata insufficient |
| fig1a, fig1c (three claims), ed2a, ed2b, ed2d, fig4c, fig4d, ed8hi, fig4e, ed4d | dandiset | blocked: data missing |

How the analysis differs from the paper's. No claim on the dandiset's own data could be
tested beyond the link: the movies carry no ROIs, stimulus times or stimulus order. The tested
claims use the figshare recordings the claims file names. For the in vivo recordings the call's
timing and ISI come from the recorded microphone channel, and the stimulus sets hold the legends'
ISIs (125, 175, 275, 575 ms). The Methods' inclusion rule ("at least five significant
prototype-evoked E/IPSCs") is not defined well enough to apply, so every listed cell is used. Dams
and EVs cannot be separated. Slice peaks are measured 2 to 30 ms after each pulse, a choice made
here. Operant presses are reconstructed from touch times printed at a resolution of 1 to 100 ms.
For the oxytocin slices the Dunn comparison family is read as five planned comparisons.

What was found about the dataset.

- The dandiset holds only raw, unaligned two-photon movies. There are no ROIs or traces (the paper
  drew ROIs by hand), no stimulus times and no order of the calls within a recording, so none of
  the 20 imaging findings can be tested. The figshare imaging items hold the same movies as TIFF.
- The behavioural data (retrieval, y-maze) and the vocalization recordings behind Fig. 1a are in
  neither the dandiset nor the figshare collection.
- Three recordings are deposited under both NV58 and NV63, with the same start times and frames.
- EV56 and EV62 are described as NV56 and NV62 in their files; their experience group is unclear.
  Indicator and location are "unknown" in every file, though the descriptions state both.
- The paper's figshare DOI does not resolve; the dandiset links the figshare collection.
- In the in vivo whole-cell data, file 19424013 is listed at -70 mV but was recorded at a
  depolarized potential; the experienced recordings are about twice as noisy as the naive ones,
  and only 3 of 26 cells show a trial-averaged EPSC locked to the call. One listed file of each of
  the in vivo naive and slice articles is not on figshare. The spreadsheets do not mark dams.
- The operant touch times are printed with five significant digits, so after 1000 s of a session
  they step by 0.1 s.